# Curve Bootstrapping, Swap Pricing & Sensitivity Analysis

This notebook:
1. Bootstraps **SOFR**, **TermSOFR3m**, **ICP**, and **CLP/USD collateral** curves
2. Prices a **5Y SOFR IRS**, a **TermSOFR3m basis swap**, and a **CLP ICP OIS swap**
3. Computes the **CLP/USD cross-currency basis** from the collateral curve
4. Computes **DV01 / sensitivity** per pillar by bumping exposed `SimpleQuote` objects

In [1]:
import json
import ORE as ore
import pandas as pd

# check: https://github.com/jmelo11/curveengine
from curveengine import CurveEngine
import plotly.graph_objects as go

## Load Configuration & Bootstrap All Curves

Curve configuration for `CurveEngine` loaded from `sensitivity_config.json` and QS results from `qs_results.json`

In [2]:
with open("sensitivity_config.json") as f:
    config = json.load(f)

engine = CurveEngine(config)

curve_names = ["SOFR", "TermSOFR3m", "ICP", "Collateral(CLP/USD)"]
curves = {}
for name in curve_names:
    curves[name] = {
        'curve': engine.getCurve(name),
        'index': engine.getIndex(name),
        'quotes': engine.getQuotes(name)
    }

refDate = ore.Date(11, 11, 2025)
ore.Settings.instance().evaluationDate = refDate
print(f"Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)")

Curves built   : SOFR, TermSOFR3m, ICP, Collateral(CLP/USD)


In [3]:
with open('qs_results.json') as file:
    qs_results = json.load(file)

ore_data = {}
dc = ore.Actual360()
for key,vals in curves.items():
    curve = vals['curve']
    nodes = curve.nodes()
    ore_data[key] = {
        'Date': [d.ISO() for d,_ in nodes],
        'DF (ORE)': [d for _,d in nodes]
    }

qs_data = {}
dc = ore.Actual360()
for curve in qs_results['curves']:
    nodes = curve['nodes']
    qs_data[curve['name'] ] = {
        'Date': [d['date'] for d in nodes],
        'DF (QS)': [d['discount_factor'] for d in nodes],
    }

compare = {}
for k in ore_data.keys():
    ore_df = pd.DataFrame(ore_data[k])
    qs_df = pd.DataFrame(qs_data[k])
    compare[k] = ore_df.merge(qs_df, right_on='Date', left_on='Date', how='outer', suffixes=['_ore','_qs'])
    compare[k]['Abs. Diff.'] = compare[k]['DF (ORE)'] - compare[k]['DF (QS)']

In [4]:
fig = go.Figure()
for k in compare.keys():
    fig.add_trace(go.Scatter(x=compare[k]['Date'], y=compare[k]['DF (ORE)'], mode='lines+markers', name=k+' (ORE)'))
    fig.add_trace(go.Scatter(x=compare[k]['Date'], y=compare[k]['DF (QS)'], mode='lines+markers', name=k+' (QS)'))

fig.update_layout(title='Absolute Difference in Discount Factors (ORE vs QS)', xaxis_title='Date', yaxis_title='Absolute Difference in DF')
fig.show()

## 1. SOFR Swap Comparison

In [5]:
calendar   = ore.NullCalendar()
convention = ore.Following
notional   = 10_000_000.0
start      = calendar.advance(refDate, ore.Period("2D"))
maturity   = calendar.advance(start, ore.Period("5Y"))

sofr_curve = curves['SOFR']['curve']
sofr_index = curves['SOFR']['index']
fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)
float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                           calendar, convention, convention,
                           ore.DateGeneration.Forward, False)

sofr_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.0378, dc,
                             float_sched, sofr_index, 0.0, dc)

sofr_handle = ore.YieldTermStructureHandle(sofr_curve)
sofr_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

In [6]:
print(f"ORE SOFR OIS 5Y  |  NPV = {sofr_swap.NPV():.2f} USD")
print(f"QS SOFR OIS 5Y   |  NPV = {qs_results['products'][0]['npv']:.2f} USD")

ORE SOFR OIS 5Y  |  NPV = 342.60 USD
QS SOFR OIS 5Y   |  NPV = 342.60 USD


In [7]:
sofr_tenors  = ["1D","3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
sofr_quotes  = curves['SOFR']['quotes']
bump = 1e-4

qs_product = next(p for p in qs_results['products'] if p['label'] == 'SOFR OIS 5Y Swap')
qs_sens = qs_product['sensitivities']

sofr_swap_rows = []
for i, q in enumerate(sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = sofr_swap.NPV()
    rq.setValue(orig - bump)
    v_down = sofr_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    sofr_swap_rows.append({"Pillar": qs_sens[i]['pillar'], "DV01 (ORE)": round(dv01, 2)})

qs_swap_rows = [{"Pillar": s['pillar'], "DV01 (QS)": round(s['dv01'], 2)} for s in qs_sens]

ore_sensitivities = pd.DataFrame(sofr_swap_rows)
qs_sensitivities = pd.DataFrame(qs_swap_rows)
sensitivity_df = ore_sensitivities.merge(qs_sensitivities, on='Pillar')

fig = go.Figure()
fig.add_trace(go.Bar(x=sensitivity_df['Pillar'], y=sensitivity_df['DV01 (ORE)'], name='SOFR Swap DV01 (ORE)'))
fig.add_trace(go.Bar(x=sensitivity_df['Pillar'], y=sensitivity_df['DV01 (QS)'], name='SOFR Swap DV01 (QS)'))

fig.update_layout(
    title='DV01 of SOFR Swap (ORE vs QS)',
    xaxis_title='Pillar',
    yaxis_title='DV01 (USD)',
    barmode='group'
)
fig.show()

sensitivity_df

,Pillar,DV01 (ORE),DV01 (QS)
0,FixedRateDeposit_USD_SOFR_1D,2.75,2.75
1,OIS_USD_SOFR_3M,2.78,2.78
2,OIS_USD_SOFR_6M,0.10,0.10
3,OIS_USD_SOFR_1Y,0.00,0.00
4,OIS_USD_SOFR_2Y,-0.00,-0.00
5,OIS_USD_SOFR_3Y,-0.00,-0.00
6,OIS_USD_SOFR_4Y,-0.00,-0.00
7,OIS_USD_SOFR_5Y,-4555.18,-4555.18
8,OIS_USD_SOFR_7Y,-17.69,-17.69
9,OIS_USD_SOFR_10Y,0.00,0.00


## 2. Term SOFR 3m Swap Comparison

In [8]:
ts3m_index = curves['TermSOFR3m']['index']
ts3m_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, notional,
                             fixed_sched, 0.04, dc,
                             float_sched, ts3m_index, 0.0, dc)
ts3m_swap.setPricingEngine(ore.DiscountingSwapEngine(sofr_handle))

qs_ts3m = next(p for p in qs_results['products'] if p['label'] == 'TermSOFR3m 5Y Swap')
print(f"ORE TermSOFR3m 5Y  |  NPV = {ts3m_swap.NPV():,.2f} USD")
print(f"QS  TermSOFR3m 5Y  |  NPV = {qs_ts3m['npv']:,.2f} USD")

ORE TermSOFR3m 5Y  |  NPV = 88,754.68 USD
QS  TermSOFR3m 5Y  |  NPV = 88,754.68 USD


In [9]:
ts3m_tenors  = ["3M","6M","1Y","2Y","3Y","4Y","5Y","7Y","10Y","15Y","20Y","30Y"]
ts3m_quotes  = curves['TermSOFR3m']['quotes']

qs_product = next(p for p in qs_results['products'] if p['label'] == 'TermSOFR3m 5Y Swap')
qs_sens = qs_product['sensitivities']

ts3m_swap_rows = []
idx = 0

# Bump TermSOFR3m (forecast curve)
for q in ts3m_quotes:
    rq = q.get('spread', q.get('rate'))
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = ts3m_swap.NPV()
    rq.setValue(orig - bump)
    v_down = ts3m_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    ts3m_swap_rows.append({"Pillar": qs_sens[idx]['pillar'], "DV01 (ORE)": round(dv01, 2)})
    idx += 1

# Bump SOFR (discount curve)
for q in sofr_quotes:
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = ts3m_swap.NPV()
    rq.setValue(orig - bump)
    v_down = ts3m_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    ts3m_swap_rows.append({"Pillar": qs_sens[idx]['pillar'], "DV01 (ORE)": round(dv01, 2)})
    idx += 1

qs_swap_rows = [{"Pillar": s['pillar'], "DV01 (QS)": round(s['dv01'], 2)} for s in qs_sens]

ore_sensitivities = pd.DataFrame(ts3m_swap_rows)
qs_sensitivities = pd.DataFrame(qs_swap_rows)
sensitivity_df = ore_sensitivities.merge(qs_sensitivities, on='Pillar')

fig = go.Figure()
fig.add_trace(go.Bar(x=sensitivity_df['Pillar'], y=sensitivity_df['DV01 (ORE)'], name='TermSOFR3m Swap DV01 (ORE)'))
fig.add_trace(go.Bar(x=sensitivity_df['Pillar'], y=sensitivity_df['DV01 (QS)'], name='TermSOFR3m Swap DV01 (QS)'))

fig.update_layout(
    title='DV01 of TermSOFR3m Swap (ORE vs QS)',
    xaxis_title='Pillar',
    yaxis_title='DV01 (USD)',
    barmode='group'
)
fig.show()

sensitivity_df

,Pillar,DV01 (ORE),DV01 (QS)
0,FixedRateDeposit_USD_TermSOFR3m_3M,5.56,5.56
1,BasisSwap_USD_SOFR_TermSOFR3m_6M,-0.01,-0.01
2,BasisSwap_USD_SOFR_TermSOFR3m_1Y,0.00,0.00
3,BasisSwap_USD_SOFR_TermSOFR3m_2Y,-0.00,-0.00
4,BasisSwap_USD_SOFR_TermSOFR3m_3Y,-0.00,-0.00
5,BasisSwap_USD_SOFR_TermSOFR3m_4Y,-0.00,-0.00
6,BasisSwap_USD_SOFR_TermSOFR3m_5Y,-4576.89,-4576.89
7,BasisSwap_USD_SOFR_TermSOFR3m_7Y,-17.69,-17.69
8,BasisSwap_USD_SOFR_TermSOFR3m_10Y,0.00,0.00
9,BasisSwap_USD_SOFR_TermSOFR3m_15Y,0.00,0.00


## 3. CLP ICP OIS Swap Comparison

In [10]:
clp_notional = 5_000_000_000.0
clp_fixed_rate = 0.0440
fx_spot     = 935.0

icp_index = curves['ICP']['index']
coll_curve = curves['Collateral(CLP/USD)']['curve']
clp_fixed_sched = ore.Schedule(start, maturity, ore.Period(ore.Semiannual),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)
clp_float_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                               calendar, convention, convention,
                               ore.DateGeneration.Forward, False)

icp_swap = ore.VanillaSwap(ore.VanillaSwap.Receiver, clp_notional,
                            clp_fixed_sched, clp_fixed_rate, dc,
                            clp_float_sched, icp_index, 0.0, dc)

# Discount with CLP/USD collateral curve
coll_handle = ore.YieldTermStructureHandle(coll_curve)
icp_swap.setPricingEngine(ore.DiscountingSwapEngine(coll_handle))

qs_icp = next(p for p in qs_results['products'] if p['label'] == 'CLP ICP OIS 5Y Swap (USD collateral)')
print(f"ORE CLP ICP OIS 5Y  |  NPV = {icp_swap.NPV()/fx_spot:,.2f} USD")
print(f"QS  CLP ICP OIS 5Y  |  NPV = {qs_icp['npv']:,.2f} USD")

ORE CLP ICP OIS 5Y  |  NPV = 4.44 USD
QS  CLP ICP OIS 5Y  |  NPV = 4.44 USD


In [11]:
fx_spot = 935.0
icp_tenors  = ["1D","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
icp_quotes  = curves['ICP']['quotes']
coll_tenors = ["1M","3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y"]
coll_quotes = curves['Collateral(CLP/USD)']['quotes']

qs_product = next(p for p in qs_results['products'] if p['label'] == 'CLP ICP OIS 5Y Swap (USD collateral)')
qs_sens = qs_product['sensitivities']

icp_swap_rows = []
idx = 0

# Bump ICP (forecast curve)
for q in icp_quotes:
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = icp_swap.NPV()
    rq.setValue(orig - bump)
    v_down = icp_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2 / fx_spot
    icp_swap_rows.append({"Pillar": qs_sens[idx]['pillar'], "DV01 (ORE)": round(dv01, 2)})
    idx += 1

# Bump Collateral(CLP/USD) (discount curve)
for q in coll_quotes:
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = icp_swap.NPV()
    rq.setValue(orig - bump)
    v_down = icp_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2 / fx_spot
    icp_swap_rows.append({"Pillar": qs_sens[idx]['pillar'], "DV01 (ORE)": round(dv01, 2)})
    idx += 1

# Bump SOFR (affects collateral curve via xccy helpers)
for q in sofr_quotes:
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = icp_swap.NPV()
    rq.setValue(orig - bump)
    v_down = icp_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2 / fx_spot
    icp_swap_rows.append({"Pillar": qs_sens[idx]['pillar'], "DV01 (ORE)": round(dv01, 2)})
    idx += 1

qs_swap_rows = [{"Pillar": s['pillar'], "DV01 (QS)": round(s['dv01'], 2)} for s in qs_sens]

ore_sensitivities = pd.DataFrame(icp_swap_rows)
qs_sensitivities = pd.DataFrame(qs_swap_rows)
sensitivity_df = ore_sensitivities.merge(qs_sensitivities, on='Pillar')

fig = go.Figure()
fig.add_trace(go.Bar(x=sensitivity_df['Pillar'], y=sensitivity_df['DV01 (ORE)'], name='ICP Swap DV01 (ORE)'))
fig.add_trace(go.Bar(x=sensitivity_df['Pillar'], y=sensitivity_df['DV01 (QS)'], name='ICP Swap DV01 (QS)'))

fig.update_layout(
    title='DV01 of CLP ICP OIS Swap (ORE vs QS)',
    xaxis_title='Pillar',
    yaxis_title='DV01 (USD)',
    barmode='group'
)
fig.show()

sensitivity_df

,Pillar,DV01 (ORE),DV01 (QS)
0,FixedRateDeposit_CLP_ICP_1D,1.46,1.46
1,FixedRateDeposit_CLP_ICP_3M,-0.94,-0.94
2,FixedRateDeposit_CLP_ICP_6M,1.13,1.13
3,OIS_CLP_ICP_1Y,1.24,1.24
4,OIS_CLP_ICP_2Y,0.31,0.31
5,OIS_CLP_ICP_3Y,0.62,0.62
6,OIS_CLP_ICP_5Y,-2370.12,-2370.12
7,OIS_CLP_ICP_7Y,-9.18,-9.18
8,OIS_CLP_ICP_10Y,0.00,0.00
9,OIS_CLP_ICP_20Y,0.00,0.00


## 4. Cross-Currency Swap Comparison

In [12]:
usd_not     = 10_000_000.0
clp_not     = usd_not * fx_spot
xccy_spread = 0.0050  # 50 bp on CLP (receive) leg

icp_curve = curves['ICP']['curve']
xccy_sched = ore.Schedule(start, maturity, ore.Period(ore.Quarterly),
                          calendar, convention, convention,
                          ore.DateGeneration.Forward, False)

# Pay USD SOFR flat, Receive CLP ICP + 50 bp
xccy_swap = ore.CrossCcyBasisSwap(
    usd_not,  ore.USDCurrency(), xccy_sched, sofr_index, 0.0, 1.0,
    clp_not,  ore.CLPCurrency(), xccy_sched, icp_index, xccy_spread, 1.0,
)

coll_handle = ore.YieldTermStructureHandle(coll_curve)
fx_quote = ore.SimpleQuote(1/fx_spot)
xccy_engine = ore.CrossCcySwapEngine(
    ore.USDCurrency(), sofr_handle,
    ore.CLPCurrency(), coll_handle,
    ore.QuoteHandle(fx_quote)
)
xccy_swap.setPricingEngine(xccy_engine)

qs_xccy = next(p for p in qs_results['products'] if p['label'] == 'Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR)')
print(f"ORE CLP/USD XCCY 5Y  |  NPV = {xccy_swap.NPV():,.2f} USD")
print(f"QS  CLP/USD XCCY 5Y  |  NPV = {qs_xccy['npv']:,.2f} USD")

ORE CLP/USD XCCY 5Y  |  NPV = 158,661.26 USD
QS  CLP/USD XCCY 5Y  |  NPV = 158,661.26 USD


In [13]:
xccy_swap_rows = []

# Bump ICP (CLP forecast curve)
for tenor, q in zip(icp_tenors, icp_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = xccy_swap.NPV()
    rq.setValue(orig - bump)
    v_down = xccy_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    xccy_swap_rows.append({"Curve": "ICP", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump CLP_COLLUSD (collateral / discount for CLP leg)
for tenor, q in zip(coll_tenors, coll_quotes):
    if 'fxPoints' in q:
        rq = q['fxPoints']
    elif 'rate' in q:
        rq = q['rate']
    else:
        continue
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = xccy_swap.NPV()
    rq.setValue(orig - bump)
    v_down = xccy_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    xccy_swap_rows.append({"Curve": "CLP_COLLUSD", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump SOFR (USD discount)
for tenor, q in zip(sofr_tenors, sofr_quotes):
    rq = q['rate']
    orig = rq.value()
    rq.setValue(orig + bump)
    v_up = xccy_swap.NPV()
    rq.setValue(orig - bump)
    v_down = xccy_swap.NPV()
    rq.setValue(orig)
    dv01 = (v_up - v_down) / 2
    xccy_swap_rows.append({"Curve": "SOFR", "Tenor": tenor, "DV01 (USD/bp)": round(dv01, 2)})

# Bump FX spot (USD/CLP)
orig_fx = fx_quote.value()
fx_quote.setValue(orig_fx + bump)
v_up = xccy_swap.NPV()
fx_quote.setValue(orig_fx - bump)
v_down = xccy_swap.NPV()
fx_quote.setValue(orig_fx)
dv01_fx = (v_up - v_down) / 2
xccy_swap_rows.append({"Curve": "FX", "Tenor": "Spot", "DV01 (USD/bp)": round(dv01_fx, 2)})

xccy_sensitivity_df = pd.DataFrame(xccy_swap_rows)

def map_qs_pillar_to_curve_tenor(pillar):
    if pillar.startswith("FixedRateDeposit_CLP_ICP_") or pillar.startswith("OIS_CLP_ICP_"):
        return "ICP", pillar.split("_")[-1]
    if pillar.startswith("FxForwardPoints_USDCLP_") or pillar.startswith("FixFloatCrossCurrencySwap_CLP_SOFR_USD_"):
        return "CLP_COLLUSD", pillar.split("_")[-1]
    if pillar.startswith("FixedRateDeposit_USD_SOFR_") or pillar.startswith("OIS_USD_SOFR_"):
        return "SOFR", pillar.split("_")[-1]
    if pillar == "USD/CLP":
        return "FX", "Spot"
    return None, None

qs_rows = []
qs_sens = qs_results['products'][3]['sensitivities']
for s in qs_sens:
    curve_name, tenor_name = map_qs_pillar_to_curve_tenor(s["pillar"])
    if curve_name is None:
        continue
    qs_rows.append({
        "Curve": curve_name,
        "Tenor": tenor_name,
        "Pillar": s["pillar"],
        "DV01 (USD/bp)": round(s["dv01"], 2)
    })

qs_sensitivities = pd.DataFrame(qs_rows)
merged_df = xccy_sensitivity_df.merge(
    qs_sensitivities,
    on=["Curve", "Tenor"],
    how="outer",
    suffixes=("_ORE", "_QS")
)

# Sort by tenor duration
tenor_order = {"1D": 0, "1W": 1, "2W": 2, "1M": 3, "2M": 4, "3M": 5, "6M": 6,
               "9M": 7, "1Y": 8, "2Y": 9, "3Y": 10, "4Y": 11, "5Y": 12,
               "7Y": 13, "10Y": 14, "15Y": 15, "20Y": 16, "30Y": 17, "Spot": 18}
merged_df["_sort"] = merged_df["Tenor"].map(tenor_order)
merged_df = merged_df.sort_values(["Pillar"]).drop(columns="_sort").reset_index(drop=True)

merged_df["Abs. Diff."] = (merged_df["DV01 (USD/bp)_ORE"] - merged_df["DV01 (USD/bp)_QS"]).abs().round(2)
merged_df[["Pillar", "Tenor", "DV01 (USD/bp)_ORE", "DV01 (USD/bp)_QS", "Abs. Diff."]]

,Pillar,Tenor,DV01 (USD/bp)_ORE,DV01 (USD/bp)_QS,Abs. Diff.
0,FixFloatCrossCurrencySwap_CLP_SOFR_USD_10Y,10Y,0.00,0.00,0.00
1,FixFloatCrossCurrencySwap_CLP_SOFR_USD_1Y,1Y,-1.80,-1.80,0.00
2,FixFloatCrossCurrencySwap_CLP_SOFR_USD_20Y,20Y,0.00,0.00,0.00
3,FixFloatCrossCurrencySwap_CLP_SOFR_USD_2Y,2Y,-5.10,-5.10,0.00
4,FixFloatCrossCurrencySwap_CLP_SOFR_USD_3Y,3Y,-13.20,-13.20,0.00
5,FixFloatCrossCurrencySwap_CLP_SOFR_USD_5Y,5Y,-4447.42,-4447.42,0.00
6,FixFloatCrossCurrencySwap_CLP_SOFR_USD_7Y,7Y,-17.18,-17.18,0.00
7,FixedRateDeposit_CLP_ICP_1D,1D,-2.73,-2.73,0.00
8,FixedRateDeposit_CLP_ICP_3M,3M,1.75,1.75,0.00
9,FixedRateDeposit_CLP_ICP_6M,6M,-2.11,-2.11,0.00
